# LeetCode 1267 — Count Servers that Communicate bằng Python

Link đề: https://leetcode.com/problems/count-servers-that-communicate/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Union-Find, Matrix, Counting

Skeleton LeetCode:

```python
class Solution:
    def countServers(self, grid: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Dịch điều kiện "giao tiếp được với **ít nhất một** máy khác" thành phép đếm: **hàng hoặc cột của nó có ≥ 2 máy**.
2. Đi từ **quét hàng/cột cho từng máy** `O(mn(m+n))` → **đếm trước theo hàng và cột** `O(mn)`.
3. Thấy bài dưới góc nhìn **đồ thị**: máy là đỉnh, cùng hàng/cột là cạnh ⇒ Union-Find / DFS, đếm máy trong các thành phần cỡ ≥ 2.
4. Dùng thành thạo `sum`, `zip(*grid)` (chuyển vị), `Counter` cho bài lưới.
5. Tránh lỗi: đếm trùng máy vừa có hàng vừa có cột tốt, so sánh `>= 1` (tự đếm chính mình), đếm số hàng/cột thay vì số máy.

## 2. Đề bài & ví dụ

Cho bản đồ trung tâm máy chủ là ma trận `grid` kích thước `m × n`; `1` là có máy chủ, `0` là không. Hai máy **giao tiếp** được nếu nằm **cùng hàng** hoặc **cùng cột**.

Trả về **số máy chủ giao tiếp được với ít nhất một máy khác**.

**Ví dụ 1:** `grid = [[1,0],[0,1]]` → `0` (hai máy khác hàng, khác cột).

**Ví dụ 2:** `grid = [[1,0],[1,1]]` → `3`.

**Ví dụ 3:** `grid = [[1,1,0,0],[0,0,1,0],[0,0,1,0],[0,0,0,1]]` → `4` (hai máy hàng 0; hai máy cột 2; máy góc dưới phải bị cô lập).

**Ràng buộc:** `1 <= m, n <= 250`, `grid[i][j] ∈ {0, 1}`.

Lưu ý: "giao tiếp" chỉ cần **cùng hàng hoặc cùng cột**, không cần kề nhau và không bị máy ở giữa chặn.

In [1]:
import random
import time
from collections import Counter, deque
from typing import List

EX1 = [[1, 0], [0, 1]]
EX2 = [[1, 0], [1, 1]]
EX3 = [[1, 1, 0, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1]]
examples = [(EX1, 0), (EX2, 3), (EX3, 4)]

def show(grid):
    for row in grid:
        print(" ".join("S" if v else "." for v in row))

for g, want in examples:
    show(g); print("-> mong đợi", want, "\n")

S .
. S
-> mong đợi 0 

S .
S S
-> mong đợi 3 

S S . .
. . S .
. . S .
. . . S
-> mong đợi 4 



## 3. Kiến thức nền

### 3.1 Đếm theo hàng: `sum(row)`

Vì ô chỉ là `0`/`1`, số máy trên hàng `i` chính là `sum(grid[i])`.

In [2]:
rows = [sum(r) for r in EX3]
print("Số máy mỗi hàng:", rows)

Số máy mỗi hàng: [2, 1, 1, 1]


### 3.2 Đếm theo cột: `zip(*grid)` (chuyển vị)

`zip(*grid)` ghép phần tử thứ `j` của mọi hàng ⇒ lần lượt cho từng **cột**. Đây là cách chuyển vị ma trận ngắn gọn trong Python.

In [3]:
print("Các cột:", list(zip(*EX3)))
cols = [sum(c) for c in zip(*EX3)]
print("Số máy mỗi cột:", cols)

Các cột: [(1, 0, 0, 0), (1, 0, 0, 0), (0, 1, 1, 0), (0, 0, 0, 1)]
Số máy mỗi cột: [1, 1, 2, 1]


### 3.3 Danh sách toạ độ máy & `Counter`

Khi lưới thưa, có thể chỉ lưu toạ độ các máy. `Counter(r for r, c in servers)` đếm số máy theo hàng.

In [4]:
servers = [(i, j) for i, row in enumerate(EX3) for j, v in enumerate(row) if v]
print("Máy:", servers)
print("Theo hàng:", Counter(i for i, _ in servers))
print("Theo cột :", Counter(j for _, j in servers))

Máy: [(0, 0), (0, 1), (1, 2), (2, 2), (3, 3)]
Theo hàng: Counter({0: 2, 1: 1, 2: 1, 3: 1})
Theo cột : Counter({2: 2, 0: 1, 1: 1, 3: 1})


### 3.4 Biểu thức điều kiện gộp với `or` và `sum` của boolean

Trong Python `True == 1`, nên `sum(điều_kiện for ...)` đếm số phần tử thoả điều kiện.

In [5]:
vals = [3, 0, 5, 1]
print(sum(v >= 2 for v in vals), "phần tử >= 2")
print(sum(1 for v in vals if v == 0 or v == 5))

2 phần tử >= 2
2


### 3.5 Union-Find tối giản

Gộp nhóm các máy liên thông (cùng hàng/cột, bắc cầu). `find` có nén đường đi, `size` theo dõi kích thước nhóm.

In [6]:
parent = list(range(5)); size = [1] * 5
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        if size[ra] < size[rb]: ra, rb = rb, ra
        parent[rb] = ra; size[ra] += size[rb]
union(0, 1); union(1, 2)
print([find(x) for x in range(5)], "kích thước nhóm của 0:", size[find(0)])

[0, 0, 0, 3, 4] kích thước nhóm của 0: 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**:

```
S S . .     hàng 0: 2 máy
. . S .     hàng 1: 1 máy
. . S .     hàng 2: 1 máy
. . . S     hàng 3: 1 máy
cột: 1 1 2 1
```

Xét từng máy:
- (0,0): hàng 0 có 2 máy ⇒ giao tiếp ✔
- (0,1): hàng 0 có 2 máy ⇒ ✔
- (1,2): hàng 1 chỉ 1 máy, nhưng cột 2 có 2 máy ⇒ ✔
- (2,2): cột 2 có 2 máy ⇒ ✔
- (3,3): hàng 3 có 1, cột 3 có 1 ⇒ ✘ (cô lập)

Đáp án: **4**. Một máy cô lập ⇔ nó là máy **duy nhất** trên hàng **và** duy nhất trên cột.

In [7]:
rows = [sum(r) for r in EX3]
cols = [sum(c) for c in zip(*EX3)]
for i, j in servers:
    ok = rows[i] >= 2 or cols[j] >= 2
    print(f"máy ({i},{j}): hàng có {rows[i]}, cột có {cols[j]} -> {'giao tiếp' if ok else 'CÔ LẬP'}")

máy (0,0): hàng có 2, cột có 1 -> giao tiếp
máy (0,1): hàng có 2, cột có 1 -> giao tiếp
máy (1,2): hàng có 1, cột có 2 -> giao tiếp
máy (2,2): hàng có 1, cột có 2 -> giao tiếp
máy (3,3): hàng có 1, cột có 1 -> CÔ LẬP


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi máy, quét cả hàng và cả cột xem có **máy khác** không.

In [8]:
def count_brute(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if not grid[i][j]:
                continue
            found = False
            for k in range(n):
                if k != j and grid[i][k]:
                    found = True; break
            if not found:
                for k in range(m):
                    if k != i and grid[k][j]:
                        found = True; break
            ans += found
    return ans

for g, want in examples:
    print(count_brute(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


**Độ phức tạp:** mỗi máy quét `O(m + n)` ô; có tới `m·n` máy ⇒ cận trên **`O(m·n·(m + n))`** ≈ `250·250·500 = 3.1·10^7` thao tác (thực tế nhờ `break` sớm thường ít hơn nhiều, xem mục 11). Bộ nhớ `O(1)`.

**Nút thắt:** hàng `i` bị quét lại cho **mỗi** máy trên hàng đó. Số máy của một hàng/cột không đổi ⇒ đếm **một lần** rồi tra cứu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** máy `(i, j)` giao tiếp được ⇔ `rows[i] >= 2` **hoặc** `cols[j] >= 2`, với `rows[i]`, `cols[j]` là số máy trên hàng `i`, cột `j` (tính cả chính nó).

Tương đương: đáp án = **tổng số máy − số máy cô lập**, với máy cô lập là máy có `rows[i] == 1` **và** `cols[j] == 1`.

**Quy tắc "nếu… thì…":**
- Nếu hàng có ≥ 2 máy ⇒ mọi máy trên hàng đó đều được đếm.
- Nếu hàng có đúng 1 máy ⇒ máy đó được đếm khi và chỉ khi cột của nó có ≥ 2 máy.
- Nếu hàng có 0 máy ⇒ bỏ qua.

**Góc nhìn đồ thị:** nối các máy cùng hàng/cột. Máy giao tiếp được ⇔ nó có **ít nhất một cạnh** ⇔ thành phần liên thông chứa nó có ≥ 2 đỉnh. (Lưu ý: đề chỉ cần giao tiếp **trực tiếp**; nhưng "có ≥ 1 hàng xóm" và "thành phần ≥ 2" là tương đương, nên Union-Find cũng đúng.)

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm hàng/cột (lời giải chuẩn)

1. Lượt 1: tính `rows[i]`, `cols[j]`.
2. Lượt 2: đếm các máy có `rows[i] > 1 or cols[j] > 1`.

In [9]:
def count_rowcol(grid):
    m, n = len(grid), len(grid[0])
    rows = [0] * m
    cols = [0] * n
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                rows[i] += 1
                cols[j] += 1
    return sum(1 for i in range(m) for j in range(n)
               if grid[i][j] and (rows[i] > 1 or cols[j] > 1))

for g, want in examples:
    print(count_rowcol(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


Bản nộp LeetCode của Cách 2 (dùng `zip(*grid)` cho gọn):

In [10]:
class SolutionRowCol:
    def countServers(self, grid: list[list[int]]) -> int:
        rows = [sum(r) for r in grid]
        cols = [sum(c) for c in zip(*grid)]
        return sum(
            1
            for i, row in enumerate(grid)
            for j, v in enumerate(row)
            if v and (rows[i] > 1 or cols[j] > 1)
        )

print([SolutionRowCol().countServers(g) for g, _ in examples])

[0, 3, 4]


**Độ phức tạp Cách 2:** hai lượt quét ⇒ **`O(m·n)` thời gian**, **`O(m + n)` bộ nhớ**.

### 7.2 Cách 3 — Tổng trừ số máy cô lập, xử lý theo hàng

Với mỗi hàng: nếu có ≥ 2 máy ⇒ cộng hết; nếu có đúng 1 máy ở cột `j` ⇒ cộng 1 khi `cols[j] > 1`. Cách này chỉ cần biết **vị trí máy duy nhất** của hàng — tiện khi hàng được cho dưới dạng danh sách vị trí.

In [11]:
def count_by_row(grid):
    cols = [sum(c) for c in zip(*grid)]
    ans = 0
    for row in grid:
        k = sum(row)
        if k >= 2:
            ans += k
        elif k == 1:
            j = row.index(1)
            if cols[j] >= 2:
                ans += 1
    return ans

for g, want in examples:
    print(count_by_row(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


**Độ phức tạp Cách 3:** `O(m·n)` thời gian, `O(n)` bộ nhớ. Hằng số nhỏ hơn Cách 2 vì `sum(row)` và `row.index` chạy ở tốc độ C.

### 7.3 Cách 4 — Lưu toạ độ máy (tốt cho lưới thưa)

Liệt kê máy một lần, đếm bằng `Counter`, rồi chỉ duyệt danh sách máy. Nếu input cho sẵn dạng danh sách toạ độ (`S` máy, lưới rất lớn), cách này `O(S)` thay vì `O(m·n)`.

In [12]:
def count_sparse(grid):
    servers = [(i, j) for i, row in enumerate(grid) for j, v in enumerate(row) if v]
    rc = Counter(i for i, _ in servers)
    cc = Counter(j for _, j in servers)
    return sum(1 for i, j in servers if rc[i] > 1 or cc[j] > 1)

for g, want in examples:
    print(count_sparse(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


**Độ phức tạp Cách 4:** `O(m·n)` để liệt kê (hoặc `O(S)` nếu đã có danh sách), `O(S)` bộ nhớ.

### 7.4 Cách 5 — Union-Find theo hàng/cột (góc nhìn đồ thị)

Kỹ thuật hay: thay vì nối máy với máy (có thể `O(S²)` cạnh), tạo **đỉnh ảo cho mỗi hàng** (`0..m-1`) và **mỗi cột** (`m..m+n-1`). Mỗi máy `(i, j)` ⇒ `union(i, m + j)`. Một thành phần chứa `k` máy với `k ≥ 2` ⇒ cả `k` máy giao tiếp được.

In [13]:
def count_union_find(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m + n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                a, b = find(i), find(m + j)
                if a != b:
                    parent[a] = b
    comp = Counter(find(i) for i in range(m) for j in range(n) if grid[i][j])
    return sum(k for k in comp.values() if k >= 2)

for g, want in examples:
    print(count_union_find(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


Vì sao đúng: mỗi máy `(i, j)` là một "cạnh" nối đỉnh hàng `i` với đỉnh cột `j`. Nếu máy cô lập (`rows[i] = cols[j] = 1`) thì đỉnh `i` và `m+j` không có cạnh nào khác ⇒ thành phần của nó chứa **đúng 1 máy**. Ngược lại nếu `rows[i] ≥ 2` (hoặc `cols[j] ≥ 2`) thì một máy khác cũng chạm đỉnh `i` (hoặc `m+j`) ⇒ thành phần chứa ≥ 2 máy. Vậy "thành phần có ≥ 2 máy" ⇔ "máy không cô lập".

**Độ phức tạp Cách 5:** `O(m·n·α(m+n))` thời gian, `O(m + n)` bộ nhớ (cộng `Counter` theo gốc).

**(Thêm) BFS trên đồ thị máy:** từ mỗi máy chưa thăm, BFS sang mọi máy cùng hàng/cột. Để kịp thời gian, mỗi hàng/cột chỉ "mở" một lần.

In [14]:
def count_bfs(grid):
    m, n = len(grid), len(grid[0])
    row_servers = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    col_servers = [[i for i in range(m) if grid[i][j]] for j in range(n)]
    seen = set(); row_done = [False] * m; col_done = [False] * n
    ans = 0
    for si in range(m):
        for sj in row_servers[si]:
            if (si, sj) in seen:
                continue
            seen.add((si, sj)); q = deque([(si, sj)]); size = 0
            while q:
                i, j = q.popleft(); size += 1
                if not row_done[i]:
                    row_done[i] = True
                    for jj in row_servers[i]:
                        if (i, jj) not in seen:
                            seen.add((i, jj)); q.append((i, jj))
                if not col_done[j]:
                    col_done[j] = True
                    for ii in col_servers[j]:
                        if (ii, j) not in seen:
                            seen.add((ii, j)); q.append((ii, j))
            if size >= 2:
                ans += size
    return ans

for g, want in examples:
    print(count_bfs(g), "| mong đợi", want)

0 | mong đợi 0
3 | mong đợi 3
4 | mong đợi 4


## 8. TRACE từng bước

Trace **Cách 2** trên Ví dụ 2 `[[1,0],[1,1]]`:

| Bước | Trạng thái |
|---|---|
| lượt 1, ô (0,0) | `rows = [1, 0]`, `cols = [1, 0]` |
| lượt 1, ô (1,0) | `rows = [1, 1]`, `cols = [2, 0]` |
| lượt 1, ô (1,1) | `rows = [1, 2]`, `cols = [2, 1]` |
| lượt 2, (0,0) | `rows[0]=1`, `cols[0]=2` ⇒ ✔, `ans = 1` |
| lượt 2, (1,0) | `rows[1]=2` ⇒ ✔, `ans = 2` |
| lượt 2, (1,1) | `rows[1]=2` ⇒ ✔, `ans = 3` |

In [15]:
def trace_rowcol(grid):
    m, n = len(grid), len(grid[0])
    rows, cols = [0] * m, [0] * n
    print("Lượt 1 — đếm:")
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                rows[i] += 1; cols[j] += 1
                print(f"   máy ({i},{j}): rows={rows}, cols={cols}")
    print("Lượt 2 — kiểm tra:")
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                ok = rows[i] > 1 or cols[j] > 1
                ans += ok
                print(f"   ({i},{j}): rows[{i}]={rows[i]}, cols[{j}]={cols[j]} -> {'✔' if ok else '✘'}  ans={ans}")
    return ans

print("Kết quả:", trace_rowcol(EX2))

Lượt 1 — đếm:
   máy (0,0): rows=[1, 0], cols=[1, 0]
   máy (1,0): rows=[1, 1], cols=[2, 0]
   máy (1,1): rows=[1, 2], cols=[2, 1]
Lượt 2 — kiểm tra:
   (0,0): rows[0]=1, cols[0]=2 -> ✔  ans=1
   (1,0): rows[1]=2, cols[0]=2 -> ✔  ans=2
   (1,1): rows[1]=2, cols[1]=1 -> ✔  ans=3
Kết quả: 3


In [16]:
def trace_union_find(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m + n))
    name = lambda x: f"H{x}" if x < m else f"C{x - m}"
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                a, b = find(i), find(m + j)
                if a != b:
                    parent[a] = b
                print(f"máy ({i},{j}): nối {name(i)} - {name(m + j)}; gốc hiện tại: " +
                      ", ".join(f"{name(x)}->{name(find(x))}" for x in range(m + n)))
    comp = Counter(find(i) for i in range(m) for j in range(n) if grid[i][j])
    print("Số máy theo thành phần:", {name(k): v for k, v in comp.items()})
    return sum(v for v in comp.values() if v >= 2)

print("Kết quả:", trace_union_find(EX3))

máy (0,0): nối H0 - C0; gốc hiện tại: H0->C0, H1->H1, H2->H2, H3->H3, C0->C0, C1->C1, C2->C2, C3->C3
máy (0,1): nối H0 - C1; gốc hiện tại: H0->C1, H1->H1, H2->H2, H3->H3, C0->C1, C1->C1, C2->C2, C3->C3
máy (1,2): nối H1 - C2; gốc hiện tại: H0->C1, H1->C2, H2->H2, H3->H3, C0->C1, C1->C1, C2->C2, C3->C3
máy (2,2): nối H2 - C2; gốc hiện tại: H0->C1, H1->C2, H2->C2, H3->H3, C0->C1, C1->C1, C2->C2, C3->C3
máy (3,3): nối H3 - C3; gốc hiện tại: H0->C1, H1->C2, H2->C2, H3->C3, C0->C1, C1->C1, C2->C2, C3->C3
Số máy theo thành phần: {'C1': 2, 'C2': 2, 'C3': 1}
Kết quả: 4


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def countServers(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        rows = [0] * m
        cols = [0] * n
        for i in range(m):
            for j in range(n):
                if grid[i][j]:
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] and (rows[i] > 1 or cols[j] > 1):
                    ans += 1
        return ans

for g, want in examples:
    got = Solution().countServers(g)
    print(got, "OK" if got == want else "SAI")

0 OK
3 OK
4 OK


## 10. Vì sao thuật toán đúng

**Bổ đề.** Máy `s = (i, j)` giao tiếp được với ít nhất một máy khác ⇔ `rows[i] ≥ 2` hoặc `cols[j] ≥ 2`.

- (⇒) Nếu `s` giao tiếp với `t ≠ s` thì `t` cùng hàng `i` (⇒ hàng `i` có ít nhất `s` và `t`, `rows[i] ≥ 2`) hoặc cùng cột `j` (⇒ `cols[j] ≥ 2`).
- (⇐) Nếu `rows[i] ≥ 2` thì ngoài `s` còn ít nhất một máy khác trên hàng `i` ⇒ `s` giao tiếp được. Tương tự với cột.

Cách 2 tính đúng `rows`, `cols` ở lượt 1 (mỗi máy cộng 1 vào đúng hàng và cột của nó), rồi đếm đúng các máy thoả bổ đề ở lượt 2. Mỗi máy được xét **đúng một lần** ⇒ không đếm trùng. ∎

**Cách 3** đếm theo hàng: hàng có `k ≥ 2` máy ⇒ cả `k` máy thoả bổ đề; hàng có 1 máy ⇒ thoả ⇔ `cols[j] ≥ 2`; hàng rỗng ⇒ 0. Cộng theo các hàng rời nhau ⇒ đúng tổng.

## 11. Phân tích Time / Space

Gọi `m, n ≤ 250`, `S` = số máy (≤ `m·n` = 62 500).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(m·n·(m+n))` (cận trên) | `O(1)` |
| 2. Đếm hàng/cột | `O(m·n)` | `O(m + n)` |
| 3. Theo hàng | `O(m·n)` | `O(n)` |
| 4. Toạ độ + Counter | `O(m·n)` (hoặc `O(S)` nếu input thưa) | `O(S)` |
| 5. Union-Find hàng/cột | `O(m·n·α)` | `O(m + n)` |
| (+) BFS | `O(m·n)` | `O(S + m + n)` |

Brute force: cận trên `O(m·n·(m+n))` là bi quan. Máy cô lập phải quét hết hàng + cột, nhưng mỗi hàng có tối đa 1 máy cô lập nên phần này chỉ `O(m·(m+n))`; máy có bạn thì `break` khi gặp bạn. Thực tế (mục 12.3) brute force chạy khá nhanh ở 250×250 — nhưng Cách 2 vẫn đơn giản và có cận `O(m·n)` chắc chắn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "brute": count_brute,
    "rowcol": count_rowcol,
    "rowcol_submit": lambda g: SolutionRowCol().countServers(g),
    "by_row": count_by_row,
    "sparse": count_sparse,
    "union_find": count_union_find,
    "bfs": count_bfs,
    "submit": lambda g: Solution().countServers(g),
}
edge = [
    ([[0]], 0), ([[1]], 0),                         # 1 ô
    ([[1, 1]], 2), ([[1], [1]], 2),                 # 1 hàng / 1 cột
    ([[1, 0, 1]], 2),                               # không kề nhau vẫn giao tiếp
    ([[0, 0], [0, 0]], 0),
    ([[1, 1], [1, 1]], 4),
    ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 0),         # đường chéo: tất cả cô lập
    ([[1, 0, 0, 1], [0, 0, 0, 0], [0, 1, 0, 0]], 2),
]
for name, f in ALL.items():
    for g, want in examples + edge:
        snapshot = [r[:] for r in g]
        assert f(g) == want, (name, g, f(g), want)
        assert g == snapshot
print("Mọi cách đúng với", len(examples + edge), "test, không sửa input")

Mọi cách đúng với 12 test, không sửa input


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [19]:
rng = random.Random(1267)
for _ in range(500):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    p = rng.choice([0.05, 0.2, 0.5, 0.9])
    g = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    want = count_brute(g)
    for name, f in ALL.items():
        assert f(g) == want, (name, g)
print("500 lưới ngẫu nhiên: mọi cách khớp brute force")

500 lưới ngẫu nhiên: mọi cách khớp brute force


### 12.3 Giới hạn 250 × 250

In [20]:
diag = [[1 if i == j else 0 for j in range(250)] for i in range(250)]   # mọi máy cô lập
dense = [[1 if rng.random() < 0.5 else 0 for _ in range(250)] for _ in range(250)]
sparse = [[1 if rng.random() < 0.003 else 0 for _ in range(250)] for _ in range(250)]
for label, g in [("đường chéo", diag), ("dày 50%", dense), ("thưa 0.3%", sparse)]:
    res = {}
    for name, f in ALL.items():
        t0 = time.perf_counter()
        res[name] = f(g)
        res[name + "_t"] = time.perf_counter() - t0
    vals = {res[k] for k in ALL}
    assert len(vals) == 1
    print(f"{label:11}: đáp án {vals.pop():6} | brute {res['brute_t']:.3f}s | rowcol {res['rowcol_submit_t']:.3f}s | uf {res['union_find_t']:.3f}s")

đường chéo : đáp án      0 | brute 0.006s | rowcol 0.004s | uf 0.005s


dày 50%    : đáp án  31435 | brute 0.015s | rowcol 0.008s | uf 0.029s


thưa 0.3%  : đáp án    118 | brute 0.005s | rowcol 0.004s | uf 0.004s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng "máy trên hàng tốt" + "máy trên cột tốt" (đếm trùng)

Máy nằm trên cả hàng có ≥ 2 máy và cột có ≥ 2 máy bị đếm hai lần.

In [21]:
def count_wrong1(grid):
    rows = [sum(r) for r in grid]
    cols = [sum(c) for c in zip(*grid)]
    return sum(k for k in rows if k > 1) + sum(k for k in cols if k > 1)   # SAI: trùng

print("EX2 — sai:", count_wrong1(EX2), "| đúng:", Solution().countServers(EX2))
print("[[1,1],[1,1]] — sai:", count_wrong1([[1, 1], [1, 1]]), "| đúng:", Solution().countServers([[1, 1], [1, 1]]))

EX2 — sai: 4 | đúng: 3
[[1,1],[1,1]] — sai: 8 | đúng: 4


### Lỗi 2 — So sánh `>= 1` thay vì `>= 2` (tự đếm chính mình)

`rows[i]` đã bao gồm chính máy `(i, j)`, nên `rows[i] >= 1` luôn đúng với mọi máy.

In [22]:
def count_wrong2(grid):
    rows = [sum(r) for r in grid]
    cols = [sum(c) for c in zip(*grid)]
    return sum(1 for i, r in enumerate(grid) for j, v in enumerate(r)
               if v and (rows[i] >= 1 or cols[j] >= 1))              # SAI

print("EX1 — sai:", count_wrong2(EX1), "| đúng:", Solution().countServers(EX1))
print("EX3 — sai:", count_wrong2(EX3), "| đúng:", Solution().countServers(EX3))

EX1 — sai: 2 | đúng: 0
EX3 — sai: 5 | đúng: 4


### Lỗi 3 — Dùng `and` thay vì `or`

Đòi máy phải có bạn **cả** trên hàng **lẫn** trên cột — quá chặt.

In [23]:
def count_wrong3(grid):
    rows = [sum(r) for r in grid]
    cols = [sum(c) for c in zip(*grid)]
    return sum(1 for i, r in enumerate(grid) for j, v in enumerate(r)
               if v and rows[i] > 1 and cols[j] > 1)                 # SAI: and

print("EX3 — sai:", count_wrong3(EX3), "| đúng:", Solution().countServers(EX3))

EX3 — sai: 0 | đúng: 4


### Lỗi 4 — Chỉ xét máy **kề** (4 hướng) như bài đếm đảo

Bê nguyên khuôn flood-fill 4 hướng: hai máy cùng hàng nhưng cách nhau ô trống bị coi là không giao tiếp.

In [24]:
def count_wrong4(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    a, b = i + di, j + dj
                    if 0 <= a < m and 0 <= b < n and grid[a][b]:
                        ans += 1; break                              # SAI: chỉ xét ô kề
    return ans

g = [[1, 0, 1], [0, 0, 0], [1, 0, 0]]
show(g)
print("sai:", count_wrong4(g), "| đúng:", Solution().countServers(g))

S . S
. . .
S . .
sai: 0 | đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"cùng hàng hoặc cùng cột"** ⇒ đếm theo hàng/cột trước (`rows`, `cols`), không cần đồ thị.
- **"ít nhất một cái khác"** ⇒ so với `>= 2` khi số đếm đã gồm chính nó.
- Đáp án = tổng − số phần tử "cô lập" là một cách nghĩ ngược hữu ích.
- `zip(*grid)` để lấy cột; `sum(row)` khi ô là 0/1.
- Union-Find với **đỉnh ảo cho hàng và cột** là mẹo tránh `O(S²)` cạnh — dùng lại trong bài 947 (Most Stones Removed) và 1559.

## 15. Biến thể của bài

1. **Đếm số cặp máy giao tiếp được**: `Σ C(rows[i], 2) + Σ C(cols[j], 2)` (cặp cùng hàng và cùng cột không thể trùng nhau vì hai máy khác nhau không thể vừa cùng hàng vừa cùng cột).
2. **Nhóm lan truyền** (giao tiếp gián tiếp) — số nhóm / nhóm lớn nhất ⇒ Union-Find (Cách 5).
3. **LeetCode 947 — Most Stones Removed with Same Row or Column**: đáp án = số đá − số thành phần.
4. **LeetCode 531/533 — Lonely Pixel**: đếm điểm ảnh đen **duy nhất** trên hàng và cột — chính là số máy cô lập!
5. **Giao tiếp theo cả đường chéo**: thêm đếm theo `i − j` và `i + j`.

Minh hoạ biến thể 1, 2 và 4:

In [25]:
def count_pairs(grid):
    rows = [sum(r) for r in grid]
    cols = [sum(c) for c in zip(*grid)]
    return sum(k * (k - 1) // 2 for k in rows) + sum(k * (k - 1) // 2 for k in cols)

def largest_group(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m + n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                parent[find(i)] = find(m + j)
    comp = Counter(find(i) for i in range(m) for j in range(n) if grid[i][j])
    return max(comp.values(), default=0), len(comp)

def lonely(grid):
    total = sum(map(sum, grid))
    return total - Solution().countServers(grid)

print("Số cặp EX3:", count_pairs(EX3), "| (nhóm lớn nhất, số nhóm) EX3:", largest_group(EX3), "| máy cô lập EX3:", lonely(EX3))

Số cặp EX3: 2 | (nhóm lớn nhất, số nhóm) EX3: (2, 3) | máy cô lập EX3: 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mỗi máy quét hàng & cột | `O(mn(m+n))` | `O(1)` | Kiểm chứng |
| 2. Đếm hàng/cột | `rows[i] > 1 or cols[j] > 1` | `O(mn)` | `O(m+n)` | Mặc định |
| 3. Theo hàng | Hàng ≥ 2 cộng hết; hàng = 1 xét cột | `O(mn)` | `O(n)` | Nhanh nhất thực tế, ít bộ nhớ |
| 4. Toạ độ + Counter | Chỉ làm việc trên danh sách máy | `O(S)` sau khi liệt kê | `O(S)` | Input thưa / cho dạng toạ độ |
| 5. Union-Find hàng/cột | Đỉnh ảo hàng & cột, đếm thành phần ≥ 2 máy | `O(mn·α)` | `O(m+n)` | Khi cần thông tin nhóm (biến thể) |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().countServers([[1,0,0,1],[0,1,1,0],[0,0,0,0],[1,0,0,0]])` = ?

### Bài 2 (tính tay)
Với lưới dưới, viết `rows`, `cols` và đánh dấu máy cô lập:
```
1 0 0
0 0 1
0 1 0
0 1 0
```

### Bài 3 (điền TODO)
Hoàn thiện `count_isolated(grid)`: trả về **số máy cô lập**, dùng tối đa `O(n)` bộ nhớ.

### Bài 4 (tự viết lại)
Viết lại Cách 2 nhưng **chỉ với một lượt quét lưới** cộng thêm một lượt duyệt qua "máy duy nhất của mỗi hàng" (gợi ý: lưu vị trí máy cuối cùng của từng hàng): `count_one_pass(grid)`.

### Bài 5 (mini challenge)
Viết `count_servers_diag(grid)`: máy giao tiếp nếu cùng hàng, cùng cột, **hoặc cùng đường chéo** (`i − j` bằng nhau) hoặc cùng đường chéo phụ (`i + j` bằng nhau).

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_isolated(grid):
    # TODO: cols = số máy theo cột; với mỗi hàng có đúng 1 máy ở cột j, nếu cols[j] == 1 thì đếm
    pass

# Bài 4 — TODO
def count_one_pass(grid):
    # TODO: một lượt: rows[i], cols[j], last_col[i]
    # TODO: sau đó: hàng >= 2 cộng hết, hàng == 1 kiểm tra cols[last_col[i]]
    pass

# Bài 5 — TODO
def count_servers_diag(grid):
    # TODO: đếm theo 4 khoá: hàng, cột, i-j, i+j
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `rows = [2, 2, 0, 1]`, `cols = [2, 1, 1, 1]`. (2) Máy (3,0): hàng 1 máy nhưng cột 0 có 2 ⇒ ✔. (3) Tất cả 5 máy đều ✔ ⇒ **5**.

**Bài 2.** (1) `rows = [1, 1, 1, 1]`, `cols = [1, 2, 1]`. (2) Máy (0,0) và (1,2) cô lập. (3) Đáp án `countServers` = 2.

**Bài 3.** (1) Máy cô lập ⇔ hàng có đúng 1 máy **và** cột có đúng 1 máy. (2) `row.index(1)` cho vị trí. (3) Bộ nhớ chỉ là mảng `cols`.

**Bài 4.** (1) Trong lượt quét, cập nhật `last_col[i] = j` mỗi khi gặp máy. (2) Hàng có đúng 1 máy thì `last_col[i]` chính là vị trí máy đó. (3) Lượt sau chỉ duyệt `m` hàng, không duyệt lại lưới.

**Bài 5.** (1) Dùng 4 `Counter` (hoặc mảng cỡ `m+n−1` cho đường chéo, dịch `i − j + n − 1`). (2) Máy giao tiếp ⇔ một trong 4 số đếm ≥ 2. (3) Kiểm chứng bằng brute force so từng cặp máy.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().countServers([[1, 0, 0, 1], [0, 1, 1, 0], [0, 0, 0, 0], [1, 0, 0, 0]]) == 5
# Bài 2
B2 = [[1, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 0]]
assert [sum(r) for r in B2] == [1, 1, 1, 1] and [sum(c) for c in zip(*B2)] == [1, 2, 1]
assert Solution().countServers(B2) == 2

# Bài 3
def count_isolated_sol(grid):
    cols = [sum(c) for c in zip(*grid)]
    return sum(1 for row in grid if sum(row) == 1 and cols[row.index(1)] == 1)

# Bài 4
def count_one_pass_sol(grid):
    m, n = len(grid), len(grid[0])
    rows, cols, last = [0] * m, [0] * n, [-1] * m
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                rows[i] += 1; cols[j] += 1; last[i] = j
    ans = 0
    for i in range(m):
        if rows[i] >= 2:
            ans += rows[i]
        elif rows[i] == 1 and cols[last[i]] >= 2:
            ans += 1
    return ans

# Bài 5
def count_servers_diag_sol(grid):
    s = [(i, j) for i, r in enumerate(grid) for j, v in enumerate(r) if v]
    keys = [Counter(i for i, j in s), Counter(j for i, j in s),
            Counter(i - j for i, j in s), Counter(i + j for i, j in s)]
    return sum(1 for i, j in s
               if keys[0][i] > 1 or keys[1][j] > 1 or keys[2][i - j] > 1 or keys[3][i + j] > 1)

def count_servers_diag_brute(grid):
    s = [(i, j) for i, r in enumerate(grid) for j, v in enumerate(r) if v]
    return sum(1 for a in s if any(b != a and (a[0] == b[0] or a[1] == b[1] or
               a[0] - a[1] == b[0] - b[1] or a[0] + a[1] == b[0] + b[1]) for b in s))

rng = random.Random(4)
for _ in range(300):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    g = [[1 if rng.random() < 0.3 else 0 for _ in range(n)] for _ in range(m)]
    total = sum(map(sum, g))
    assert count_isolated_sol(g) == total - count_brute(g)
    assert count_one_pass_sol(g) == count_brute(g)
    assert count_servers_diag_sol(g) == count_servers_diag_brute(g)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm theo hàng/cột":**

```python
rows = [sum(r) for r in grid]
cols = [sum(c) for c in zip(*grid)]
ans = sum(1 for i, r in enumerate(grid) for j, v in enumerate(r)
          if v and (rows[i] > 1 or cols[j] > 1))
```

**Template Union-Find đỉnh ảo hàng/cột:** đỉnh `0..m-1` cho hàng, `m..m+n-1` cho cột; mỗi điểm `(i, j)` ⇒ `union(i, m + j)`.

**Checklist trước khi submit:**
- [ ] Điều kiện là `> 1` (hoặc `>= 2`), vì số đếm đã gồm chính máy đó.
- [ ] Dùng `or` giữa hàng và cột.
- [ ] Mỗi máy chỉ được đếm một lần (không cộng riêng hàng và cột).
- [ ] Giao tiếp không cần kề nhau.
- [ ] Test: lưới 1 ô, đường chéo (đáp án 0), lưới toàn 1.
- [ ] Đúng chữ ký: `countServers(self, grid: list[list[int]]) -> int`.